# Phase 3: Acoustic Feature Engineering Walkthrough
### Converting Respiratory Sounds into Numbers for Machine Learning

Welcome to the **Feature Engineering** notebook! In this notebook, we explore the **307 numerical acoustic features** extracted from each respiratory cycle in the ICBHI 2017 dataset.

> **💡 The Goal in Plain English:**  
> Machine learning algorithms cannot directly understand raw audio waves. We need to convert each breath into meaningful numbers (features) that describe **how the sound feels, how high-pitched it is, how loud it is, and whether it has sudden pops or musical whistling**.

---

### 🎧 The 4 Types of Lung Sounds We Are Detecting:
1. 🟢 **Normal (52.8%)**: Smooth, clean breath sounds with no blockages.
2. 🟠 **Crackle (27.0%)**: Short, sudden popping clicks (like bubble wrap or Velcro), common in Pneumonia.
3. 🔵 **Wheeze (12.8%)**: Continuous high-pitched musical whistling caused by narrowed airways (COPD, Asthma).
4. 🟣 **Both (7.3%)**: Severe breath cycles containing both crackles and wheezes together.

---

### 🔬 What Do the 307 Features Measure?
* **Timbre / Texture (MFCCs - 240 features)**: Captures the overall frequency envelope and "timbre" of the sound (like telling a guitar apart from a piano).
* **Pitch & Brightness (Spectral - 38 features)**: Tells us if the sound is high-pitched (like a wheeze) or low-pitched (normal breath).
* **Pops & Loudness (Temporal - 9 features)**: Detects sharp clicks, energy bursts, and duration of the breath.
* **Musical Notes (Chroma - 24 features)**: Measures musical pitch harmonics (wheezes sound like musical notes).

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import scipy.signal as signal
import scipy.stats as stats
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import f_classif

# Setup clean plotting style
%matplotlib inline
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams['figure.dpi'] = 120
print("All libraries imported successfully!")

## 📌 Step 1: Load the Datasets
We load the clean feature matrices generated by `scripts/extract_features.py`:
* `X_train.csv` / `y_train.csv`: Training data (4,142 cycles across 79 patients)
* `X_test.csv` / `y_test.csv`: Testing data (2,756 cycles across 49 patients)
* `feature_names.json`: Dictionary listing all 307 feature names

In [ ]:
data_dir = "../data/processed"

# Load datasets
X_train = pd.read_csv(os.path.join(data_dir, "X_train.csv"))
y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))
meta_train = pd.read_csv(os.path.join(data_dir, "metadata_train.csv"))

X_test = pd.read_csv(os.path.join(data_dir, "X_test.csv"))
y_test = pd.read_csv(os.path.join(data_dir, "y_test.csv"))
meta_test = pd.read_csv(os.path.join(data_dir, "metadata_test.csv"))

with open(os.path.join(data_dir, "feature_names.json"), "r") as f:
    feature_info = json.load(f)

print(f"✓ Total Features Extracted : {X_train.shape[1]}")
print(f"✓ Training Breath Cycles    : {X_train.shape[0]:,} cycles (from {meta_train['patient_id'].nunique()} patients)")
print(f"✓ Testing Breath Cycles     : {X_test.shape[0]:,} cycles (from {meta_test['patient_id'].nunique()} patients)")
print(f"✓ Total Dataset Cycles      : {len(X_train) + len(X_test):,}")

# Class Breakdown Table
class_summary = pd.DataFrame({
    "Train Count": y_train["class"].value_counts(),
    "Train (%)": (y_train["class"].value_counts(normalize=True)*100).round(1),
    "Test Count": y_test["class"].value_counts(),
    "Test (%)": (y_test["class"].value_counts(normalize=True)*100).round(1)
})
display(class_summary)

## 📌 Step 2: What Are the 307 Features?
Let's see how many features belong to each acoustic domain.

In [ ]:
families = feature_info["feature_families"]
family_names = {
    "mfcc_stats": "MFCCs (Static Sound Shape)",
    "mfcc_delta_stats": "MFCC Delta (Speed of Sound Change)",
    "mfcc_delta2_stats": "MFCC Delta2 (Sound Acceleration)",
    "spectral_descriptors": "Spectral (Pitch, Rolloff & Brightness)",
    "temporal_dynamics": "Temporal (Pops, Clicks & Volume)",
    "chroma_tonality": "Chroma (Musical Pitch & Harmonics)"
}
counts = pd.Series({family_names.get(k, k): len(v) for k, v in families.items()})

plt.figure(figsize=(9, 4))
ax = sns.barplot(x=counts.values, y=counts.index, palette="Blues_r", edgecolor="black", linewidth=0.7)
for p in ax.patches:
    ax.annotate(f"{int(p.get_width())} features", (p.get_width() + 2, p.get_y() + p.get_height() / 2),
                va='center', fontsize=9, fontweight='bold')
plt.xlim(0, max(counts.values) + 20)
plt.title("Breakdown of the 307 Acoustic Features by Domain", fontsize=12, fontweight="bold")
plt.xlabel("Number of Features")
plt.tight_layout()
plt.show()

## 📌 Step 3: Listen & Look at a Breath Sound
What does a breath sound look like to a computer?
* **Waveform**: Shows volume over time.
* **Spectrogram**: A heat map showing which frequencies (pitches) are active over time.
* **MFCCs**: A compressed visual fingerprint of the sound texture.

In [ ]:
# Simulate a 2.5-second exemplar breath with wheezing and crackles
sr = 16000
duration = 2.5
t = np.linspace(0, duration, int(duration * sr), endpoint=False)

# Background breathing sound
audio = 0.3 * np.random.randn(len(t)) * np.sin(2 * np.pi * 0.4 * t)**2
# Add a wheeze (a musical whistle at 400 Hz between 0.8s and 1.8s)
audio += 0.3 * np.sin(2 * np.pi * 400 * t) * (t > 0.8) * (t < 1.8)
# Add crackles (sharp clicks at specific moments)
for click in [0.5, 1.1, 1.4, 2.0]:
    pos = int(click * sr)
    audio[pos:pos+35] += np.hanning(35) * 0.8

audio = audio / np.max(np.abs(audio)) * 0.95

# Compute Spectrogram and MFCCs
S = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=1024, hop_length=512, n_mels=64)
S_db = librosa.power_to_db(S, ref=np.max)
mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20, n_fft=1024, hop_length=512)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

# 1. Waveform
axes[0].plot(t, audio, color="#1971c2", lw=1)
axes[0].set_ylabel("Volume")
axes[0].set_title("1. Audio Waveform (Look for sharp vertical clicks = Crackles)", fontweight="bold")

# 2. Spectrogram
img1 = librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis='time', y_axis='mel', ax=axes[1], cmap="magma")
axes[1].set_ylabel("Pitch (Hz)")
axes[1].set_title("2. Spectrogram (Look for horizontal glowing lines = Wheeze Whistle)", fontweight="bold")
fig.colorbar(img1, ax=axes[1], format="%+2.0f dB")

# 3. MFCCs
img2 = librosa.display.specshow(mfcc, sr=sr, hop_length=512, x_axis='time', ax=axes[2], cmap="coolwarm")
axes[2].set_ylabel("MFCC Band")
axes[2].set_title("3. MFCC Fingerprint (Texture numbers passed to ML models)", fontweight="bold")
axes[2].set_xlabel("Time (Seconds)")
fig.colorbar(img2, ax=axes[2])

plt.tight_layout()
plt.show()

## 📌 Step 4: Comparing Healthy vs. Abnormal Breaths
Let's look at boxplots of real data from the 4,142 training breaths.

> **💡 What to look for:**  
> Notice how **Spectral Rolloff** and **Spectral Centroid** are significantly higher for Wheezes and Both, because whistling contains higher pitch energy!

In [ ]:
plot_df = X_train.copy()
plot_df["class"] = y_train["class"].values

features_to_plot = [
    ("mfcc_00_mean", "1. MFCC 0 (Overall Volume / Loudness)"),
    ("mfcc_01_mean", "2. MFCC 1 (Bass vs Treble Balance)"),
    ("spectral_centroid_mean", "3. Spectral Centroid (Center Frequency / Pitch)"),
    ("spectral_rolloff_85_mean", "4. Spectral Rolloff 85% (High Pitch Boundary)"),
    ("spectral_flatness_mean", "5. Spectral Flatness (Noisiness vs Tonality)"),
    ("rms_energy_mean", "6. RMS Energy (Breath Energy)")
]

palette = {"Normal": "#2b8a3e", "Crackle": "#d9480f", "Wheeze": "#1971c2", "Both": "#7048e8"}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, (col, title) in enumerate(features_to_plot):
    ax = axes[idx]
    sns.boxplot(data=plot_df, x="class", y=col, hue="class", order=["Normal", "Crackle", "Wheeze", "Both"],
                palette=palette, ax=ax, width=0.5, fliersize=1.5, legend=False)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Value")

plt.suptitle("Acoustic Feature Distributions Across Sound Classes (Real ICBHI Data)", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 📌 Step 5: Which Features Move Together? (Correlation Heatmap)
Some features are related (e.g., if a breath is high-pitched, both Spectral Centroid and Spectral Rolloff go up). This heatmap shows the correlation between key features.

In [ ]:
corr_cols = [
    "cycle_duration_feat", "mfcc_00_mean", "mfcc_01_mean", "mfcc_02_mean",
    "spectral_centroid_mean", "spectral_bandwidth_mean", "spectral_rolloff_85_mean",
    "spectral_flatness_mean", "zcr_mean", "rms_energy_mean"
]
rename_map = {
    "cycle_duration_feat": "Duration",
    "mfcc_00_mean": "MFCC 0",
    "mfcc_01_mean": "MFCC 1",
    "mfcc_02_mean": "MFCC 2",
    "spectral_centroid_mean": "Centroid",
    "spectral_bandwidth_mean": "Bandwidth",
    "spectral_rolloff_85_mean": "Rolloff 85%",
    "spectral_flatness_mean": "Flatness",
    "zcr_mean": "Zero Crossing",
    "rms_energy_mean": "RMS Energy"
}
clean_corr = plot_df[corr_cols].rename(columns=rename_map).corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(clean_corr, dtype=bool))
sns.heatmap(clean_corr, mask=mask, cmap="vlag", vmin=-1.0, vmax=1.0,
            annot=True, fmt=".2f", square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix of Key Acoustic Features", fontsize=12, fontweight="bold", pad=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 📌 Step 6: Visualizing the Data in 2D (PCA & t-SNE)
Can a computer separate Normal breaths from Abnormal breaths? We project all 307 features down to 2 dimensions using **PCA** and **t-SNE**.

In [ ]:
# Sample 1,500 points for quick and clear plotting
sample_idx = np.random.RandomState(42).choice(len(X_train), size=1500, replace=False)
X_sub = X_train.iloc[sample_idx]
y_sub = y_train.iloc[sample_idx]

X_scaled = StandardScaler().fit_transform(X_sub)

# Run PCA and t-SNE
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
tsne_coords = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for cls_name, color in [("Normal", "#2b8a3e"), ("Crackle", "#d9480f"), ("Wheeze", "#1971c2"), ("Both", "#7048e8")]:
    mask = y_sub["class"] == cls_name
    axes[0].scatter(pca_coords[mask, 0], pca_coords[mask, 1], c=color, label=f"{cls_name} ({mask.sum()})", alpha=0.6, s=25)
    axes[1].scatter(tsne_coords[mask, 0], tsne_coords[mask, 1], c=color, label=cls_name, alpha=0.6, s=25)

axes[0].set_title(f"PCA 2D Projection ({pca.explained_variance_ratio_.sum()*100:.1f}% Variance Explained)", fontweight="bold")
axes[0].set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")
axes[0].legend(loc="upper right")

axes[1].set_title("t-SNE 2D Manifold Embedding", fontweight="bold")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].legend(loc="upper right")

plt.suptitle("2D Visualization of the 307-Dimensional Acoustic Space", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 📌 Step 7: Which Features Matter Most? (ANOVA F-Score Ranking)
We use the statistical **ANOVA F-score** to rank which features have the strongest differences between Normal and Abnormal breath sounds.

In [ ]:
f_scores, _ = f_classif(X_train, y_train["is_adventitious"])
top_ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "F_Score": f_scores
}).sort_values(by="F_Score", ascending=False).reset_index(drop=True)

top15 = top_ranking.head(15)

plt.figure(figsize=(9, 5))
sns.barplot(data=top15, x="F_Score", y="Feature", palette="mako", edgecolor="black", linewidth=0.6)
plt.title("Top 15 Most Discriminative Features (Normal vs Abnormal Breath)", fontsize=12, fontweight="bold")
plt.xlabel("ANOVA F-Score (Higher = More Useful for Classification)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:")
for i, r in top15.head(5).iterrows():
    print(f"  {i+1}. {r['Feature']} (F-Score: {r['F_Score']:.1f})")

## 📌 Step 8: Ready for Phase 4 (Machine Learning!)
We verify that our training and testing datasets are properly separated and ready for model training.

In [ ]:
train_p = set(meta_train['patient_id'])
test_p = set(meta_test['patient_id'])

print("=== DATASET HEALTH CHECK ===")
print(f"✓ Training Samples : {len(X_train):,} cycles (79 patients)")
print(f"✓ Testing Samples  : {len(X_test):,} cycles (49 patients)")
print(f"✓ Feature Columns  : {X_train.shape[1]} (307 acoustic descriptors)")
print(f"✓ Missing Values   : {X_train.isna().sum().sum()} in train, {X_test.isna().sum().sum()} in test")
print("\n🚀 All systems green! The datasets in data/processed/ are ready for Phase 4 Model Training!")